# Classifying variants with a large language model (few-shot prompting)

Here we use a **generative large language model** (LLM), like the ones behind chatbots. It can follow written instructions, so we can:

1. **describe** the categories in plain words (the *instructions*);
2. show it a few **examples** classified by us (**few-shot**: a few examples; **zero-shot**: no examples);
3. ask it to classify a new variant and to **explain** its choice.

Then we compare its answers with our own classification (the *ground truth*).

The model is too big to run on our computers or on myBinder. It runs on a server of the University of Bern, **GPUStack**: we send our questions to it over the internet, through an **API** (*Application Programming Interface*: a way for programs to talk to each other).

## Where to run this script
Our machine needs to be in the university network (on premise or through VPN) in order to connect to GPUStack. mybinder cannot do that. In order to run your script, you need a local installation of
- python
- jupyter notebooks
- pandas
- requests

See the [helpme guide](../helpme/notebooks.md).

## Connect to the university's model

We need three things:
- `API_URL`: the address of the server;
- `MODEL_NAME`: the name of the model on that server;
- the **API key**: a personal password that gives access to the server. We are using an API linked to my personal account: feel free to use it, but without overdoing it; it will automatically expire in 6 months.

Attention! The key is like a password: do not write it in the notebook and do not share it. Create a separate **.env file** for it and save it in the same folder of this notebook.

In [1]:
import requests  # Sends requests over the internet.
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
MODEL_NAME = os.getenv("MODEL_NAME")
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",  # The key, sent with every request.
    "Content-Type": "application/json",    # We send our request in JSON format.
}

## The data: our ground truth

There are [two ground truth files](https://github.com/DHBern/text-analysis-course/tree/main/session07#ground-truth).

In [2]:
import pandas as pd  # Provides DataFrames for tables.
pd.set_option("display.max_colwidth", 120)  # Shows longer texts in table cells.

GT_FILE = "gt_yours.csv"  # The file with our classification: can be gt_easy.csv or gt_yours  # CHANGE HERE
CATEGORIES = ["error", "substantive", "vocabulary"]  # The categories decided in class.

ground_truth = pd.read_csv(GT_FILE)
ground_truth["category"] = ground_truth["category"].str.strip().str.lower()  # "Error " -> "error"
ground_truth = ground_truth[ground_truth["category"].isin(CATEGORIES)]  # Drops empty or unknown categories.

print(len(ground_truth), "variants in the ground truth")
ground_truth["category"].value_counts()

60 variants in the ground truth


category
vocabulary     27
error          21
substantive    12
Name: count, dtype: int64

## Examples and test set

We split the ground truth in two groups:
- **examples**: a few variants per category, shown to the model in the instructions;
- **test set**: all the others. The model classifies them, and we check its answers against ours.

A variant used as an example must not be in the test set, otherwise we would be giving the model the answer.

In [3]:
N_EXAMPLES = 2  # Examples per category. 0 = zero-shot.

examples = ground_truth.groupby("category").sample(N_EXAMPLES, random_state=1) if N_EXAMPLES else ground_truth.iloc[:0]
test = ground_truth.drop(examples.index)

print(len(examples), "examples,", len(test), "variants in the test set")
examples[["del_full", "add_full", "category"]]

6 examples, 54 variants in the test set


,del_full,add_full,category
35,j,the,error
25,The,the,error
56,are,to discover him,substantive
19,know,have learned,substantive
52,about,About,vocabulary
41,blossomed,bloomed,vocabulary


## The instructions (the *prompt*)

The model only knows what we write here. The definitions are the ones we discussed in class.

This is the part to experiment with: change the definitions and see how the results change.

In [4]:
DEFINITIONS = """\
- error: a mistake is corrected: a typo, a misspelling, a grammatical problem.
- substantive: the meaning of the text changes (different content, facts, names, events).
- vocabulary: same meaning, better wording: a synonym, a more precise, effective or elegant word."""

def show_variant(row):
    # Describes one variant as text: the passage before and after, and the replaced words.
    return (f'Before: "{row["del_in_context"]}"\n'
            f'After:  "{row["add_in_context"]}"\n'
            f'Replaced: "{row["del_full"]}" -> "{row["add_full"]}"')

instructions = f"""You are an expert in textual scholarship. You classify revisions in the manuscript of Mary Shelley's Frankenstein.
Each revision replaces some words with others. Classify it in one of these categories:
{DEFINITIONS}

Answer in this format: category | one short sentence explaining why."""

if len(examples):
    instructions += "\n\nExamples:\n" + "\n".join(
        f'"{r["del_full"]}" -> "{r["add_full"]}": {r["category"]}' for _, r in examples.iterrows())

print(instructions)

You are an expert in textual scholarship. You classify revisions in the manuscript of Mary Shelley's Frankenstein.
Each revision replaces some words with others. Classify it in one of these categories:
- error: a mistake is corrected: a typo, a misspelling, a grammatical problem.
- substantive: the meaning of the text changes (different content, facts, names, events).
- vocabulary: same meaning, better wording: a synonym, a more precise, effective or elegant word.

Answer in this format: category | one short sentence explaining why.

Examples:
"j" -> "the": error
"The" -> "the": error
"are" -> "to discover him": substantive
"know" -> "have learned": substantive
"about" -> "About": vocabulary
"blossomed" -> "bloomed": vocabulary


## Ask the model about one variant

A request to the model contains:
- `messages`, in two parts:
  - `system`: the instructions, the same for every variant;
  - `user`: the variant to classify;
- `temperature=0`: we ask for the most likely answer, so that the results are (almost) the same each time;
- `max_tokens`: the maximum length of the answer.

The server sends back a JSON answer; the text of the model is in `["choices"][0]["message"]["content"]`.

Some models "think aloud" before answering, between `<think>` and `</think>`. We remove that part and keep only the answer.

In [5]:
import re  # Regular expressions, to remove the "thinking" part.
import time  # To wait between requests.

def ask(row):
    # Sends one variant to the model and returns its answer (a text).
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "system", "content": instructions},
                     {"role": "user", "content": show_variant(row)}],
        "temperature": 0,
        "max_tokens": 1000,
    }
    for attempt in range(5):
        try:
            response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=120)
        except requests.exceptions.RequestException as error:  # No connection, or the server is too slow.
            print("Connection problem, waiting 20 seconds:", str(error)[:100])
            time.sleep(20)
            continue
        if response.status_code == 429 or response.status_code >= 500:  # Too many requests, or server busy.
            print("Server busy, waiting 20 seconds:", response.status_code)
            time.sleep(20)
            continue
        response.raise_for_status()  # Other problems (e.g. wrong key or URL): stop and show the error.
        answer = response.json()["choices"][0]["message"]["content"] or ""
        return re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()
    return ""

def read_category(answer):
    # Extracts the category from an answer such as "vocabulary | a synonym of ...".
    first_part = answer.split("|")[0].lower()
    for category in CATEGORIES:
        if category in first_part:
            return category
    return "unknown"

row = test.iloc[0]
print(show_variant(row))
answer = ask(row)
print("\nModel:", answer)
print("Category:", read_category(answer), "| Ours:", row["category"])

Before: "fled – On seeing me he darted towards me and tore the girl from my arms hastened towards the deeper parts"
After:  "fled – On seeing me he darted towards me and tearing the girl from my arms hastened towards the deeper parts"
Replaced: "tore" -> "tearing"

Model: error | changing "tore" to "tearing" fixes the grammatical structure of the sentence.
Category: error | Ours: error


## Classify the test set

One request per variant: this takes a few seconds per variant, in total several minutes.

In [6]:
answers = []
for i, (_, row) in enumerate(test.iterrows(), start=1):
    answers.append(ask(row))
    print(i, "/", len(test), end="\r")  # Progress.

test = test.assign(answer=answers)
test["predicted"] = test["answer"].apply(read_category)
test["reason"] = test["answer"].str.split("|").str[-1].str.strip()

test[["del_full", "add_full", "category", "predicted", "reason"]]

,del_full,add_full,category,predicted,reason
0,tore,tearing,error,error,"changing ""tore"" to ""tearing"" fixes the grammatical structure of the sentence."
1,softness,gentleness,vocabulary,vocabulary,"""gentleness"" is a synonym for ""softness"" that gives the phrase a more precise and elegant tone."
2,pain,the agony of my wound,substantive,substantive,"The revision specifies the source and intensity of the pain, changing the content from general pain to the agony of ..."
3,prayers,vows,vocabulary,vocabulary,“vows” is a more solemn and effective substitute for “prayers” while preserving the sense of daily supplication or p...
4,horrors of my situation.,the outrages & the anguish I had endured.,substantive,substantive,The revision replaces a general reference to the horrors of his situation with specific outrages and anguish he had ...
5,journey,toils,substantive,substantive,"The revision changes the referent from a physical journey to labor or hardship, altering the content."
6,Geneva,that town,vocabulary,unknown,
7,gazen,gazed,error,error,The revision corrects a misspelling of “gazed.”
8,athought,an idea,vocabulary,unknown,
9,hav e,had,error,error,"The revision corrects a typographical spacing error, “hav e,” to the proper word “had.”"


## Evaluation

- **Accuracy**: share of variants where the model agrees with us.
- **Confusion matrix**: rows = our category, columns = the model's category. The diagonal holds the agreements; the other cells show *which* categories are confused.

In [7]:
accuracy = (test["predicted"] == test["category"]).mean()
print(f"Accuracy: {accuracy:.0%} on {len(test)} variants")
pd.crosstab(test["category"], test["predicted"], rownames=["ours"], colnames=["model"])

Accuracy: 59% on 54 variants


model,error,substantive,unknown,vocabulary
ours,,,,
error,17,0,1,1
substantive,0,5,1,4
vocabulary,1,4,10,10


In [8]:
# The disagreements, with the model's explanation: who is right?
test.loc[test["predicted"] != test["category"], ["del_full", "add_full", "category", "predicted", "reason"]]

,del_full,add_full,category,predicted,reason
6,Geneva,that town,vocabulary,unknown,
8,athought,an idea,vocabulary,unknown,
11,on,in,error,vocabulary,The preposition is changed for slightly different phrasing without altering the basic meaning.
12,eyes,face,vocabulary,substantive,"The revision changes the specific body part from his eyes to his face, altering the detail of the action."
15,destroy,make desolate.,substantive,vocabulary,“make desolate” is a more expressive synonym for “destroy” without changing the basic meaning.
17,black,deep,substantive,vocabulary,"""deep"" replaces ""black"" with a more evocative, less redundant descriptor while preserving the sense of dark eyes."
20,I have,At length I,vocabulary,substantive,adding “At length” introduces a temporal sequence and changes the aspect from present perfect to past.
22,createfor me.,create.,vocabulary,unknown,
26,unutterable,inextinguishable,vocabulary,substantive,It changes the meaning by describing the hatred as unquenchable rather than inexpressible.
30,return,journey,substantive,unknown,


## Save to CSV file

In [ ]:
# Saves the classified variants, with the model's answers, to a CSV file.
# The model name is part of the file name, so results of different models are kept separate.
# The ground truth file name is also included in the output file name.
OUTPUT_FILE = "variants_classified_llm_" + re.sub(r"[^A-Za-z0-9.-]", "_", MODEL_NAME) + "_" + re.sub(r"[^A-Za-z0-9.-]", "_", GT_FILE)  # "/" or ":" are not allowed in file names.
test.to_csv(OUTPUT_FILE, index=False)
print("Saved to", OUTPUT_FILE)

Saved to variants_classified_llm_qwen3.8-27b_gt_yours.csv.csv


## Things to keep in mind

- **Plausible ≠ correct.** The explanations sound convincing even when the category is wrong. Read them critically.
- **Not fully reproducible.** The same prompt can give different answers on different days, and the models offered on a server change or disappear. Always note which model you used.
- **The prompt is part of the method.** Small changes in the definitions change the results: document the prompt you used.
- **Where does the data go?** Here, to a university server. Check where a model runs before sending confidential or personal data.
- **Ground truth is a choice too.** Another person might classify some variants differently: the "right" answer is sometimes debatable.

## Exercises

1. Set `N_EXAMPLES = 0` (zero-shot) and run sections 2–6 again. Does accuracy change? Then try `N_EXAMPLES = 4`.
2. Rewrite `DEFINITIONS` (section 3), for example with more precise criteria for *vocabulary* vs *substantive*. Does accuracy improve?
3. Look at the disagreements (section 6). Are there cases where you now agree with the model rather than with the ground truth?